In [ ]:
# ============================================================
# STAGE 4 — POST-PROCESSING & VALIDATION
# D4 — Branch B — Structural Conversion
# ============================================================
#
# Validation basis:
# - Fixed Stage 1 document-grounded reference dataset
# - Preserved Branch B raw response
# - Branch B technical diagnostics
# - Branch B parsed extraction, only when structurally evaluable
# - D4 comparison rules frozen from Validation A
# ============================================================

from google.colab import files
from pathlib import Path

import hashlib
import html
import json
import re
import unicodedata

import pandas as pd

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D4"
BRANCH_ID = "B"
BRANCH_NAME = "Structural Conversion"

EXPECTED_REFERENCE_COUNT = 83

EXTRACTION_FIELDS = [
    "Section",
    "Concept Name",
    "Concept Value",
    "Publication Restricted"
]

REFERENCE_FIELDS = EXTRACTION_FIELDS + [
    "Source Location"
]

ALIGNMENT_IDENTITY_FIELDS = [
    "Section",
    "Concept Name"
]

PRIMARY_CORRECTNESS_FIELDS = [
    "Concept Value",
    "Publication Restricted"
]

OUTPUT_DIR = Path(
    "outputs_D4_validation_branch_B"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Validation configured.")
print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH_ID, "-", BRANCH_NAME)

In [ ]:
# ============================================================
# 2. Upload validation inputs
# ============================================================
# Required:
#   1) D4_reference_values.csv
#   2) D4_branch_B_raw_response.txt
#   3) D4_branch_B_technical_diagnostics.json
#
# Optional:
#   4) D4_branch_B_parsed_extraction.json

uploaded = files.upload()
uploaded_files = list(uploaded.keys())

csv_files = [
    f for f in uploaded_files
    if f.lower().endswith(".csv")
]

txt_files = [
    f for f in uploaded_files
    if f.lower().endswith(".txt")
]

json_files = [
    f for f in uploaded_files
    if f.lower().endswith(".json")
]

if len(csv_files) != 1:
    raise ValueError(
        "Upload exactly one Stage 1 reference-values CSV."
    )

if len(txt_files) != 1:
    raise ValueError(
        "Upload exactly one preserved Branch B raw-response TXT."
    )

REFERENCE_FILE = csv_files[0]
RAW_RESPONSE_FILE = txt_files[0]

TECHNICAL_DIAGNOSTICS_FILE = None
PARSED_EXTRACTION_FILE = None

for file_name in json_files:

    with open(
        file_name,
        "r",
        encoding="utf-8-sig"
    ) as f:
        obj = json.load(f)

    if not isinstance(obj, dict):
        continue

    if obj.get("document_id") != DOCUMENT_ID:
        continue

    if obj.get("branch") != BRANCH_ID:
        continue

    if (
        "structurally_evaluable" in obj
        and "record_schema_valid" in obj
        and "valid_json" in obj
    ):
        TECHNICAL_DIAGNOSTICS_FILE = file_name

    if isinstance(
        obj.get("records"),
        list
    ):
        PARSED_EXTRACTION_FILE = file_name


if TECHNICAL_DIAGNOSTICS_FILE is None:
    raise ValueError(
        "Could not identify the D4 Branch B "
        "technical diagnostics JSON."
    )

print("Reference:", REFERENCE_FILE)
print("Raw response:", RAW_RESPONSE_FILE)
print(
    "Technical diagnostics:",
    TECHNICAL_DIAGNOSTICS_FILE
)
print(
    "Parsed extraction:",
    PARSED_EXTRACTION_FILE
)

In [ ]:
# ============================================================
# 3. Load inputs and verify provenance
# ============================================================

reference_df = pd.read_csv(
    REFERENCE_FILE,
    encoding="utf-8-sig"
)

with open(
    RAW_RESPONSE_FILE,
    "r",
    encoding="utf-8"
) as f:
    raw_response_text = f.read()

with open(
    TECHNICAL_DIAGNOSTICS_FILE,
    "r",
    encoding="utf-8-sig"
) as f:
    technical_diagnostics = json.load(f)

parsed_extraction = None

if PARSED_EXTRACTION_FILE is not None:

    with open(
        PARSED_EXTRACTION_FILE,
        "r",
        encoding="utf-8-sig"
    ) as f:
        parsed_extraction = json.load(f)


if technical_diagnostics.get(
    "document_id"
) != DOCUMENT_ID:
    raise ValueError(
        "Technical diagnostics document_id "
        "does not match D4."
    )

if technical_diagnostics.get(
    "branch"
) != BRANCH_ID:
    raise ValueError(
        "Technical diagnostics branch "
        "does not match Branch B."
    )

if parsed_extraction is not None:

    if parsed_extraction.get(
        "document_id"
    ) != DOCUMENT_ID:
        raise ValueError(
            "Parsed extraction document_id "
            "does not match D4."
        )

    if parsed_extraction.get(
        "branch"
    ) != BRANCH_ID:
        raise ValueError(
            "Parsed extraction branch "
            "does not match Branch B."
        )


if len(reference_df) != EXPECTED_REFERENCE_COUNT:
    raise ValueError(
        "Unexpected D4 Stage 1 reference count: "
        f"{len(reference_df)}"
    )


missing_reference_fields = [
    field
    for field in REFERENCE_FIELDS
    if field not in reference_df.columns
]

if missing_reference_fields:
    raise ValueError(
        "Stage 1 reference is missing fields: "
        f"{missing_reference_fields}"
    )


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(8192),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


input_provenance = {
    "reference_file":
        REFERENCE_FILE,

    "reference_sha256":
        sha256_file(
            REFERENCE_FILE
        ),

    "raw_response_file":
        RAW_RESPONSE_FILE,

    "raw_response_sha256":
        sha256_file(
            RAW_RESPONSE_FILE
        ),

    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_FILE,

    "technical_diagnostics_sha256":
        sha256_file(
            TECHNICAL_DIAGNOSTICS_FILE
        )
}

if PARSED_EXTRACTION_FILE is not None:

    input_provenance[
        "parsed_extraction_file"
    ] = PARSED_EXTRACTION_FILE

    input_provenance[
        "parsed_extraction_sha256"
    ] = sha256_file(
        PARSED_EXTRACTION_FILE
    )


print(
    "Reference records:",
    len(reference_df)
)

In [ ]:
# ============================================================
# 4. Import Branch B technical/schema status
# ============================================================


raw_json_valid = False
raw_json_error = None

try:
    json.loads(
        raw_response_text
    )
    raw_json_valid = True

except json.JSONDecodeError as exc:
    raw_json_error = str(exc)


structurally_evaluable = bool(
    technical_diagnostics.get(
        "structurally_evaluable",
        False
    )
)

schema_validity = bool(
    structurally_evaluable
)

content_evaluable = bool(
    structurally_evaluable
    and parsed_extraction is not None
)


if (
    structurally_evaluable
    and parsed_extraction is None
):
    raise ValueError(
        "Branch B is marked structurally evaluable "
        "but no parsed extraction was supplied."
    )


schema_diagnostics = {
    "raw_response_valid_json":
        raw_json_valid,

    "raw_response_json_error":
        raw_json_error,

    "branch_B_valid_json":
        bool(
            technical_diagnostics.get(
                "valid_json",
                False
            )
        ),

    "record_schema_valid":
        bool(
            technical_diagnostics.get(
                "record_schema_valid",
                False
            )
        ),

    "field_types_valid":
        bool(
            technical_diagnostics.get(
                "field_types_valid",
                False
            )
        ),

    "structurally_evaluable":
        structurally_evaluable,

    "schema_validity":
        schema_validity,

    "content_evaluable":
        content_evaluable
}


print(
    json.dumps(
        schema_diagnostics,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 5. Structural-failure result when content is not evaluable
# ============================================================

if not content_evaluable:

    summary = {
        "document_id":
            DOCUMENT_ID,

        "branch":
            BRANCH_ID,

        "branch_name":
            BRANCH_NAME,

        "reference_records":
            int(
                len(reference_df)
            ),

        "extracted_records":
            None,

        "aligned_records":
            None,

        "fully_correct_records":
            None,

        "discrepant_records":
            None,

        "missing_records":
            None,

        "hallucinated_records":
            None,

        "completeness":
            None,

        "missing_rate":
            None,

        "record_precision_exact":
            None,

        "record_recall_exact":
            None,

        "record_f1_exact":
            None,

        "hallucination_rate":
            None,

        "field_accuracy":
            None,

        "schema_validity":
            schema_validity,

        "schema_diagnostics":
            schema_diagnostics,

        "structurally_evaluable":
            structurally_evaluable,

        "content_evaluable":
            False,

        "alignment_identity_fields":
            ALIGNMENT_IDENTITY_FIELDS,

        "primary_correctness_fields":
            PRIMARY_CORRECTNESS_FIELDS,

        "comparison_rules_frozen_from_branch_A":
            True,

        "validation_status":
            (
                "Structural failure — "
                "content validation not evaluable"
            ),

        "recovered_or_repaired_extraction_used":
            False,

        "input_provenance":
            input_provenance
    }


    with open(
        OUTPUT_DIR
        / "D4_branch_B_validation_summary.json",
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            summary,
            f,
            indent=2,
            ensure_ascii=False
        )


    structural_failure_df = pd.DataFrame([
        {
            "document_id":
                DOCUMENT_ID,

            "branch":
                BRANCH_ID,

            "reference_records":
                int(
                    len(reference_df)
                ),

            "raw_response_valid_json":
                raw_json_valid,

            "raw_response_json_error":
                raw_json_error,

            "schema_validity":
                schema_validity,

            "structurally_evaluable":
                structurally_evaluable,

            "content_evaluable":
                False,

            "recovered_or_repaired_extraction_used":
                False,

            "validation_status":
                (
                    "Structural failure — "
                    "content validation not evaluable"
                )
        }
    ])


    structural_failure_df.to_csv(
        OUTPUT_DIR
        / "D4_branch_B_structural_failure.csv",
        index=False
    )


    print(
        json.dumps(
            summary,
            indent=2,
            ensure_ascii=False
        )
    )

    print(
        "\nContent comparison intentionally skipped. "
        "No repaired or recovered extraction was used."
    )

In [ ]:
# ============================================================
# 6. Prepare extraction when structurally evaluable
# ============================================================

if content_evaluable:

    extracted_df = pd.DataFrame(
        parsed_extraction[
            "records"
        ]
    )

    missing_extraction_fields = [
        field
        for field in EXTRACTION_FIELDS
        if field not in extracted_df.columns
    ]

    for field in missing_extraction_fields:
        extracted_df[field] = None

    extracted_df = (
        extracted_df[
            EXTRACTION_FIELDS
        ].copy()
    )

    print(
        "Extracted records:",
        len(extracted_df)
    )

    print(
        "Missing extraction columns:",
        missing_extraction_fields
    )

else:

    extracted_df = pd.DataFrame(
        columns=EXTRACTION_FIELDS
    )

In [ ]:
# ============================================================
# 7. Frozen D4 comparison normalisation
# ============================================================


UNICODE_SPACES = {
    "\u00a0": " ",
    "\u2007": " ",
    "\u202f": " "
}

APOSTROPHE_REPLACEMENTS = {
    "\u2018": "'",
    "\u2019": "'",
    "\u02bc": "'",
    "`": "'"
}

DASH_REPLACEMENTS = {
    "\u2010": "-",
    "\u2011": "-",
    "\u2012": "-",
    "\u2013": "-",
    "\u2014": "-",
    "\u2212": "-"
}


def normalise_text(value):

    if value is None:
        return None

    text = unicodedata.normalize(
        "NFKC",
        str(value)
    )

    for source, target in (
        UNICODE_SPACES.items()
    ):
        text = text.replace(
            source,
            target
        )

    for source, target in (
        APOSTROPHE_REPLACEMENTS.items()
    ):
        text = text.replace(
            source,
            target
        )

    for source, target in (
        DASH_REPLACEMENTS.items()
    ):
        text = text.replace(
            source,
            target
        )

    text = re.sub(
        r"[ \t]+",
        " ",
        text
    )

    text = re.sub(
        r" *\n *",
        "\n",
        text
    )

    return (
        text
        .strip()
        .casefold()
    )


def normalise_html_content(value):

    if value is None:
        return None

    text = html.unescape(
        str(value)
    )

    text = re.sub(
        r"</p\s*>",
        "\n",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"<br\s*/?>",
        "\n",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"<[^>]+>",
        "",
        text
    )

    return normalise_text(
        text
    )

In [ ]:
# ============================================================
# 8. One-to-one alignment using frozen D4 identity
# ============================================================

if content_evaluable:

    ref_cmp = reference_df[
        REFERENCE_FIELDS
    ].copy(
        deep=True
    )

    ext_cmp = extracted_df.copy(
        deep=True
    )

    for df in (
        ref_cmp,
        ext_cmp
    ):

        df["_alignment_key"] = (
            df.apply(
                lambda row: (
                    normalise_text(
                        row["Section"]
                    ),
                    normalise_text(
                        row["Concept Name"]
                    )
                ),
                axis=1
            )
        )


    reference_duplicate_mask = (
        ref_cmp[
            "_alignment_key"
        ].duplicated(
            keep=False
        )
    )

    if reference_duplicate_mask.any():
        raise ValueError(
            "The fixed Stage 1 reference contains "
            "duplicate D4 alignment identities."
        )


    extraction_duplicate_mask = (
        ext_cmp[
            "_alignment_key"
        ].duplicated(
            keep="first"
        )
    )


    duplicate_extracted_records = (
        ext_cmp[
            extraction_duplicate_mask
        ].copy()
    )


    ext_unique = (
        ext_cmp[
            ~extraction_duplicate_mask
        ].copy()
    )


    validation_df = ref_cmp.merge(
        ext_unique,
        on="_alignment_key",
        how="outer",
        suffixes=(
            "_ref",
            "_ext"
        ),
        indicator=True,
        validate="one_to_one"
    )


    aligned_mask = (
        validation_df[
            "_merge"
        ] == "both"
    )

else:

    validation_df = pd.DataFrame()

    aligned_mask = pd.Series(
        dtype=bool
    )

    duplicate_extracted_records = (
        pd.DataFrame()
    )

In [ ]:
# ============================================================
# 9. Field comparison after alignment
# ============================================================


if content_evaluable:

    validation_df[
        "Section_match"
    ] = False

    validation_df[
        "Concept Name_match"
    ] = False

    validation_df[
        "Concept Value_match"
    ] = False

    validation_df[
        "Publication Restricted_match"
    ] = False

    validation_df[
        "Concept Value_normalised_match"
    ] = False

    validation_df[
        "Concept Value_content_match"
    ] = False


    validation_df.loc[
        aligned_mask,
        "Section_match"
    ] = (
        validation_df.loc[
            aligned_mask,
            "Section_ref"
        ]
        ==
        validation_df.loc[
            aligned_mask,
            "Section_ext"
        ]
    )


    validation_df.loc[
        aligned_mask,
        "Concept Name_match"
    ] = (
        validation_df.loc[
            aligned_mask,
            "Concept Name_ref"
        ]
        ==
        validation_df.loc[
            aligned_mask,
            "Concept Name_ext"
        ]
    )


    validation_df.loc[
        aligned_mask,
        "Concept Value_match"
    ] = (
        validation_df.loc[
            aligned_mask,
            "Concept Value_ref"
        ]
        ==
        validation_df.loc[
            aligned_mask,
            "Concept Value_ext"
        ]
    )


    validation_df.loc[
        aligned_mask,
        "Publication Restricted_match"
    ] = (
        validation_df.loc[
            aligned_mask,
            "Publication Restricted_ref"
        ]
        ==
        validation_df.loc[
            aligned_mask,
            "Publication Restricted_ext"
        ]
    )


    validation_df.loc[
        aligned_mask,
        "Concept Value_normalised_match"
    ] = (
        validation_df.loc[
            aligned_mask
        ].apply(
            lambda row:
                normalise_text(
                    row[
                        "Concept Value_ref"
                    ]
                )
                ==
                normalise_text(
                    row[
                        "Concept Value_ext"
                    ]
                ),
            axis=1
        )
    )


    validation_df.loc[
        aligned_mask,
        "Concept Value_content_match"
    ] = (
        validation_df.loc[
            aligned_mask
        ].apply(
            lambda row:
                normalise_html_content(
                    row[
                        "Concept Value_ref"
                    ]
                )
                ==
                normalise_html_content(
                    row[
                        "Concept Value_ext"
                    ]
                ),
            axis=1
        )
    )


    PRIMARY_MATCH_COLUMNS = [
        "Concept Value_match",
        "Publication Restricted_match"
    ]


    validation_df[
        "all_primary_fields_match"
    ] = (
        aligned_mask
        & validation_df[
            PRIMARY_MATCH_COLUMNS
        ].all(
            axis=1
        )
    )


    validation_df[
        "alignment_identity_fields_match"
    ] = (
        aligned_mask
        & validation_df[
            [
                "Section_match",
                "Concept Name_match"
            ]
        ].all(
            axis=1
        )
    )

In [ ]:
# ============================================================
# 10. Calculate content-level validation metrics
# ============================================================

if content_evaluable:

    def classify_record(row):

        if row["_merge"] == "left_only":
            return "missing"

        if row["_merge"] == "right_only":
            return "hallucinated_unmatched"

        if bool(
            row[
                "all_primary_fields_match"
            ]
        ):
            return "fully_correct"

        return "discrepant"


    validation_df[
        "record_status"
    ] = validation_df.apply(
        classify_record,
        axis=1
    )


    missing_records = (
        validation_df[
            validation_df[
                "record_status"
            ] == "missing"
        ].copy()
    )


    hallucinated_unmatched = (
        validation_df[
            validation_df[
                "record_status"
            ]
            == "hallucinated_unmatched"
        ].copy()
    )


    discrepant_records = (
        validation_df[
            validation_df[
                "record_status"
            ] == "discrepant"
        ].copy()
    )


    fully_correct_records = (
        validation_df[
            validation_df[
                "record_status"
            ] == "fully_correct"
        ].copy()
    )


    if not duplicate_extracted_records.empty:

        duplicate_extracted_records[
            "record_status"
        ] = "hallucinated_duplicate"


    N_REF = int(
        len(reference_df)
    )

    N_EXT = int(
        len(extracted_df)
    )

    N_ALIGNED = int(
        aligned_mask.sum()
    )

    N_MISSING = int(
        len(missing_records)
    )

    N_HALLUCINATED_UNMATCHED = int(
        len(
            hallucinated_unmatched
        )
    )

    N_DUPLICATE_EXTRAS = int(
        len(
            duplicate_extracted_records
        )
    )

    N_HALLUCINATED = (
        N_HALLUCINATED_UNMATCHED
        + N_DUPLICATE_EXTRAS
    )

    N_DISCREPANT = int(
        len(discrepant_records)
    )

    N_CORRECT = int(
        len(fully_correct_records)
    )


    completeness = (
        N_ALIGNED / N_REF
        if N_REF
        else 0.0
    )

    missing_rate = (
        N_MISSING / N_REF
        if N_REF
        else 0.0
    )

    record_precision = (
        N_CORRECT / N_EXT
        if N_EXT
        else 0.0
    )

    record_recall = (
        N_CORRECT / N_REF
        if N_REF
        else 0.0
    )

    record_f1 = (
        2
        * record_precision
        * record_recall
        / (
            record_precision
            + record_recall
        )
        if (
            record_precision
            + record_recall
        )
        else 0.0
    )

    hallucination_rate = (
        N_HALLUCINATED / N_EXT
        if N_EXT
        else 0.0
    )

    discrepancy_rate = (
        N_DISCREPANT
        / N_ALIGNED
        if N_ALIGNED
        else 0.0
    )


    aligned_df = (
        validation_df[
            aligned_mask
        ].copy()
    )


    field_accuracy_among_aligned = {
        "Section":
            float(
                aligned_df[
                    "Section_match"
                ].mean()
            )
            if N_ALIGNED
            else 0.0,

        "Concept Name":
            float(
                aligned_df[
                    "Concept Name_match"
                ].mean()
            )
            if N_ALIGNED
            else 0.0,

        "Concept Value":
            float(
                aligned_df[
                    "Concept Value_match"
                ].mean()
            )
            if N_ALIGNED
            else 0.0,

        "Publication Restricted":
            float(
                aligned_df[
                    "Publication Restricted_match"
                ].mean()
            )
            if N_ALIGNED
            else 0.0
    }


    correct_field_instances = int(
        aligned_df[
            PRIMARY_MATCH_COLUMNS
        ].sum().sum()
    )

    expected_field_instances = int(
        N_REF
        * len(
            PRIMARY_CORRECTNESS_FIELDS
        )
    )

    field_accuracy = (
        correct_field_instances
        / expected_field_instances
        if expected_field_instances
        else 0.0
    )

In [ ]:
# ============================================================
# 11. Field-level and HTML comparison diagnostics
# ============================================================

if content_evaluable:

    field_summary_rows = []

    for field in EXTRACTION_FIELDS:

        match_column = (
            f"{field}_match"
        )

        correct_aligned = int(
            aligned_df[
                match_column
            ].sum()
        )

        field_summary_rows.append({
            "field":
                field,

            "used_in_alignment_identity":
                field
                in ALIGNMENT_IDENTITY_FIELDS,

            "used_in_primary_correctness":
                field
                in PRIMARY_CORRECTNESS_FIELDS,

            "aligned_records_evaluated":
                N_ALIGNED,

            "correct_values_among_aligned":
                correct_aligned,

            "incorrect_values_among_aligned":
                (
                    N_ALIGNED
                    - correct_aligned
                ),

            "accuracy_among_aligned":
                (
                    correct_aligned
                    / N_ALIGNED
                    if N_ALIGNED
                    else 0.0
                ),

            "missing_expected_instances":
                N_MISSING,

            "overall_expected_instances":
                N_REF,

            "overall_accuracy_against_reference":
                (
                    correct_aligned
                    / N_REF
                    if N_REF
                    else 0.0
                )
        })


    field_summary_df = pd.DataFrame(
        field_summary_rows
    )


    html_diagnostics = pd.DataFrame([
        {
            "aligned_records":
                N_ALIGNED,

            "concept_value_exact_matches":
                int(
                    aligned_df[
                        "Concept Value_match"
                    ].sum()
                ),

            "concept_value_normalised_matches":
                int(
                    aligned_df[
                        "Concept Value_normalised_match"
                    ].sum()
                ),

            "concept_value_content_matches":
                int(
                    aligned_df[
                        "Concept Value_content_match"
                    ].sum()
                )
        }
    ])


    field_summary_df

In [ ]:
# ============================================================
# 12. Build content-validation summary
# ============================================================

if content_evaluable:

    summary = {
        "document_id":
            DOCUMENT_ID,

        "branch":
            BRANCH_ID,

        "branch_name":
            BRANCH_NAME,

        "reference_records":
            N_REF,

        "extracted_records":
            N_EXT,

        "aligned_records":
            N_ALIGNED,

        "fully_correct_records":
            N_CORRECT,

        "discrepant_records":
            N_DISCREPANT,

        "missing_records":
            N_MISSING,

        "hallucinated_records":
            N_HALLUCINATED,

        "completeness":
            round(
                completeness,
                4
            ),

        "missing_rate":
            round(
                missing_rate,
                4
            ),

        "record_precision_exact":
            round(
                record_precision,
                4
            ),

        "record_recall_exact":
            round(
                record_recall,
                4
            ),

        "record_f1_exact":
            round(
                record_f1,
                4
            ),

        "hallucination_rate":
            round(
                hallucination_rate,
                4
            ),

        "discrepancy_rate_among_aligned":
            round(
                discrepancy_rate,
                4
            ),

        "field_accuracy":
            round(
                field_accuracy,
                4
            ),

        "field_accuracy_among_aligned": {
            key:
                round(
                    value,
                    4
                )
            for key, value
            in field_accuracy_among_aligned.items()
        },

        "schema_validity":
            schema_validity,

        "schema_diagnostics":
            schema_diagnostics,

        "structurally_evaluable":
            structurally_evaluable,

        "content_evaluable":
            True,

        "alignment_identity_fields":
            ALIGNMENT_IDENTITY_FIELDS,

        "primary_correctness_fields":
            PRIMARY_CORRECTNESS_FIELDS,

        "comparison_rules_frozen_from_branch_A":
            True,

        "comparison_rules": {
            "alignment_identity":
                (
                    "Normalised Section + Concept Name"
                ),

            "primary_correctness":
                (
                    "Exact source preservation of "
                    "Concept Value and Publication Restricted"
                ),

            "concept_value_normalised_comparison":
                "Diagnostic only",

            "concept_value_html_content_comparison":
                "Diagnostic only"
        },

        "recovered_or_repaired_extraction_used":
            False,

        "input_provenance":
            input_provenance
    }


    print(
        json.dumps(
            summary,
            indent=2,
            ensure_ascii=False
        )
    )

In [ ]:
# ============================================================
# 13. Integrity checks and export validation artefacts
# ============================================================

if content_evaluable:

    assert (
        N_ALIGNED
        + N_MISSING
        == N_REF
    )

    assert (
        N_ALIGNED
        + N_HALLUCINATED_UNMATCHED
        + N_DUPLICATE_EXTRAS
        == N_EXT
    )

    assert (
        N_CORRECT
        + N_DISCREPANT
        == N_ALIGNED
    )


    for metric_name, metric_value in {
        "completeness":
            completeness,

        "missing_rate":
            missing_rate,

        "record_precision":
            record_precision,

        "record_recall":
            record_recall,

        "record_f1":
            record_f1,

        "hallucination_rate":
            hallucination_rate,

        "discrepancy_rate":
            discrepancy_rate,

        "field_accuracy":
            field_accuracy
    }.items():

        assert (
            0.0
            <= metric_value
            <= 1.0
        ), (
            f"Invalid {metric_name}: "
            f"{metric_value}"
        )


    validation_df.to_csv(
        OUTPUT_DIR
        / "D4_branch_B_validation_detailed.csv",
        index=False
    )


    missing_records.to_csv(
        OUTPUT_DIR
        / "D4_branch_B_missing_records.csv",
        index=False
    )


    hallucinated_unmatched.to_csv(
        OUTPUT_DIR
        / "D4_branch_B_hallucinated_unmatched_records.csv",
        index=False
    )


    duplicate_extracted_records.to_csv(
        OUTPUT_DIR
        / "D4_branch_B_hallucinated_duplicate_records.csv",
        index=False
    )


    discrepant_records.to_csv(
        OUTPUT_DIR
        / "D4_branch_B_discrepant_records.csv",
        index=False
    )


    fully_correct_records.to_csv(
        OUTPUT_DIR
        / "D4_branch_B_fully_correct_records.csv",
        index=False
    )


    field_summary_df.to_csv(
        OUTPUT_DIR
        / "D4_branch_B_field_error_summary.csv",
        index=False
    )


    html_diagnostics.to_csv(
        OUTPUT_DIR
        / "D4_branch_B_html_comparison_diagnostics.csv",
        index=False
    )


    with open(
        OUTPUT_DIR
        / "D4_branch_B_validation_summary.json",
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            summary,
            f,
            indent=2,
            ensure_ascii=False
        )


    print(
        "Validation integrity checks passed."
    )

    print(
        "Content-validation artefacts saved."
    )

else:

    print(
        "Structural-failure artefacts were saved. "
        "No content-level metrics or repaired extraction "
        "were fabricated."
    )

In [ ]:
# ============================================================
# 14. Download generated validation artefacts
# ============================================================

for output_file in sorted(
    OUTPUT_DIR.iterdir()
):

    if output_file.is_file():

        files.download(
            output_file
        )